In [ ]:
import requests
from requests import ConnectionError,HTTPError,Timeout
import numpy as np
import pandas as pd

url = 'https://data.ntpc.gov.tw/api/datasets/010e5b15-3823-4b20-b401-b1cf000550c5/json?page=0&size=1000'
try:
    response = requests.get(url)    
    response.raise_for_status()    
except ConnectionError:
    print('找不到伺服器')
except HTTPError:
    print('網頁找不到')
except Timeout:
    print('超過時間沒有回應')
else:
    print('沒有發生問題')

records = response.json()
#records
allRecords = pd.DataFrame(records,columns=['sna','tot','mday','sbi','sarea','ar','bemp'])
allRecords1 = allRecords.rename(columns= {'sna':'站名', 'sarea':'區域', 'ar':'地址', 'tot':'數量','sbi':'可借', 'bemp':'可還','mday':'時間'})

In [ ]:
allRecords1['站名'] = allRecords1['站名'].apply(lambda name:name[11:])

In [ ]:
allRecords1.info()

In [ ]:
allRecords1['時間'] = pd.to_datetime(allRecords1['時間'])

In [ ]:
allRecords1[['數量','可借','可還']] = allRecords1[['數量','可借','可還']].astype(int)

In [ ]:
allRecords1.info()

In [ ]:
allRecords1['時間'] = pd.to_datetime(allRecords1['時間'])

In [ ]:
allRecords1[['數量','可借','可還']] = allRecords1[['數量','可借','可還']].astype(int)

In [ ]:
allRecords1.info()

In [ ]:
import numpy as np
grouped = allRecords1.groupby('區域')
agg_df = grouped[['數量','可借','可還']].agg([("加總","sum"),("平均","mean"),("中間數","median"),("最多","max"),("最少","min")])

In [15]:
agg_df.columns.names = ["租借","統計"]
s1 = agg_df.stack(level=['租借','統計'],future_stack=True)
s1

區域   租借  統計 
三峽區  數量  加總     1540.000000
         平均       22.318841
         中間數      20.000000
         最多       63.000000
         最少       10.000000
                   ...     
金山區  可還  加總      125.000000
         平均       17.857143
         中間數      15.000000
         最多       27.000000
         最少        9.000000
Length: 315, dtype: float64

In [16]:
s1.unstack(level="統計")

統計          加總         平均   中間數    最多    最少
區域  租借                                     
三峽區 數量  1540.0  22.318841  20.0  63.0  10.0
    可借   687.0   9.956522   9.0  26.0   0.0
    可還   852.0  12.347826  10.0  40.0   3.0
三芝區 數量   170.0  21.250000  16.0  48.0  14.0
    可借    43.0   5.375000   5.0  11.0   2.0
...        ...        ...   ...   ...   ...
貢寮區 可借    22.0   7.333333   5.0  16.0   1.0
    可還    38.0  12.666667  15.0  19.0   4.0
金山區 數量   163.0  23.285714  24.0  30.0  15.0
    可借    38.0   5.428571   5.0   9.0   3.0
    可還   125.0  17.857143  15.0  27.0   9.0

[63 rows x 5 columns]